In [9]:
import aiohttp
import asyncio
from bs4 import BeautifulSoup
import json
import os
import random

CATEGORY_DIALECT_MAP = {
    "Mexican Spanish idioms": ("Mexican Spanish", 0.95, "dialect"),
    "Argentinian Spanish idioms": ("Argentinian Spanish", 0.95, "dialect"),
    "Spanish slang": ("Peninsular Spanish", 0.7, "dialect"),
    "Castillian Spanish idioms": ("Peninsular Spanish", 0.95, "standard"),
    "Australian English idioms": ("Australian English", 0.95, "dialect"),
    "American English idioms": ("American English", 0.95, "standard"),
    "British English idioms": ("British English", 0.95, "standard"),
    "Pakistani Urdu idioms": ("Pakistani Urdu", 0.95, "dialect"),
    "Hyderabadi Urdu idioms": ("Hyderabadi Urdu", 0.95, "dialect"),
    "Bhojpuri idioms": ("Bihari Hindi – Bhojpuri", 0.95, "dialect"),
    "Maithili idioms": ("Bihari Hindi – Maithili", 0.95, "dialect"),
    "Chhattisgarhi idioms": ("Eastern Hindi – Chhattisgarhi", 0.95, "dialect"),
    "Rayalaseema Telugu idioms": ("Telugu – Rayalaseema", 0.95, "dialect"),
    "Telangana Telugu idioms": ("Telugu – Telangana", 0.95, "dialect"),
    "Coastal Andhra Telugu idioms": ("Telugu – Coastal", 0.95, "dialect"),
}

LANG_DIALECT_LEXICAL_SIGNATURES = {
    "es": {  # Spanish
        "Mexican Spanish": ["órale", "güey", "chido", "no manches"],
        "Argentinian Spanish": ["che", "quilombo", "laburo", "re loco"],
        "Peninsular Spanish": ["vale", "guay", "tío"],
    },
    "en": {  # English
        "British English": ["bloody", "bugger", "bollocks", "taking the piss"],
        "Australian English": ["arvo", "mate", "bogan", "chuck a sickie"],
        "American English": ["y'all", "dude", "gonna", "freak out", "finna"],
    },
    "ur": {
        "Hyderabadi Urdu": ["nakko", "hau", "kaiku", "light le lo"],
        "Pakistani Urdu": ["scene on", "biryani scene", "chill maar"],
    },
    "hi": {
        "Awadhi": ["kahe", "tohar", "ba"],
        "Marwari": ["mharo", "tharo", "koni"],
        "Khari Boli": ["arey yaar", "tu kya keh raha hai"],
        "Bhojpuri": ["ka ho", "abey saala", "bhaiya"],
        "Maithili": ["hamra", "ehi", "tō"],
        "Chhattisgarhi": ["ka kare", "mor", "tor"],
    },
    "te": {
        "Telangana": ["baagunDi", "vedhava", "keka", "బాగుంది", "వేదవ", "కేక", "జల్ది", "దబ్బున"],
        "Coastal Andhra": ["raayabari"],
        "Rayalaseema": ["తెమలలేదా", "అట్నే", "lotta", "binne", "దబగ్గిన."],
    }
}

def match_lexical_signature(text, lang_code, full_signature_map):
    dialect_signatures = full_signature_map.get(lang_code, {})
    for dialect, clues in dialect_signatures.items():
        for word in clues:
            if word.lower() in text.lower():
                return dialect, 0.95, "dialect"
    return None, None, None


async def fetch(session, url):
    async with session.get(url, timeout=5) as response:
        return await response.text()

async def scrape_idiom_page(session, idiom_url, lang_code, counter):
    try:
        html = await fetch(session, idiom_url)
        idiom_soup = BeautifulSoup(html, 'html.parser')

        # === Gloss extraction ===
        gloss = ""
        # Prefer <ol> in definition sections
        def_lists = idiom_soup.select('ol')
        if def_lists:
            for ol in def_lists:
                items = [li.get_text(strip=True) for li in ol.find_all('li', recursive=False)]
                if items:
                    gloss = "; ".join(items)
                    break

        # === Example extraction ===
        example_list = []

        # First, look for Usage Examples headers
        for header in idiom_soup.find_all(['h3', 'h4']):
            if 'Usage examples' in header.text or 'Examples' in header.text:
                next_sibling = header.find_next_sibling()
                if next_sibling:
                    if next_sibling.name == 'ul':
                        example_list.extend([li.get_text(strip=True) for li in next_sibling.find_all('li')])
                    elif next_sibling.name == 'p':
                        example_list.append(next_sibling.get_text(strip=True))
        # Fallback: check <dl> structures
        if not example_list:
            for dl in idiom_soup.find_all('dl'):
                example_list.extend([dt.get_text(strip=True) for dt in dl.find_all('dt')])
        example = "; ".join(example_list)

        # === Category extraction ===
        category_tags = idiom_soup.select('#mw-normal-catlinks ul li')
        categories = [tag.text.strip() for tag in category_tags]

        # === Dialect detection ===
        dialect = "unspecified"
        confidence = 0.5
        dialect_type = "standard"

        for cat in categories:
            if cat in CATEGORY_DIALECT_MAP:
                dialect, confidence, dialect_type = CATEGORY_DIALECT_MAP[cat]
                break

        lex_dialect, lex_conf, lex_type = match_lexical_signature(gloss + " " + example, lang_code, LANG_DIALECT_LEXICAL_SIGNATURES)
        if lex_dialect:
            dialect, confidence, dialect_type = lex_dialect, lex_conf, lex_type

        # === ID sanitization ===
        safe_id = idiom_url.split("/")[-1].replace(' ', '_')
        safe_id = ''.join(c for c in safe_id if c.isalnum() or c in ['_', '-']).lower()

        idiom_obj = {
            "id": f"{lang_code}_{dialect.replace(' ', '_').lower()}_{str(counter).zfill(4)}",
            "idiom": idiom_url.split("/")[-1].replace('_', ' '),
            "language": lang_code,
            "dialect": dialect,
            "dialect_type": dialect_type,
            "confidence_score": confidence,
            "source": "Wiktionary",
            "source_category": "",
            "url": idiom_url,
            "categories": categories,
            "literal_meaning": "",  # Can fill manually later
            "idiomatic_meaning": gloss,
            "example": example,
            "example_count": len(example_list),
            "quality": "seed",
            "status": "pending",
            "validation_count": 0,
            "time_period": "unspecified",
            "usage_frequency": "low",
        }

        # Respectful pause to avoid overloading server
        await asyncio.sleep(0.5 + random.uniform(0, 0.5))
        return idiom_obj

    except Exception as e:
        print(f"❌ Error processing idiom '{idiom_url}': {e}")
        return None


async def scrape_wiktionary_idioms_structured(lang_label, lang_code, limit=1000):
    base_url = f"https://en.wiktionary.org/wiki/Category:{lang_label}"
    idioms = []
    seen = set()
    page_url = base_url
    counter = 1

    async with aiohttp.ClientSession() as session:
        while page_url and len(idioms) < limit:
            try:
                page_html = await fetch(session, page_url)
                soup = BeautifulSoup(page_html, 'html.parser')
                links = soup.select('.mw-category-group ul li a')
            except Exception as e:
                print(f"❌ Error fetching or parsing category page: {e}")
                break

            tasks = []
            for a in links:
                title = a.get('title')
                if not title or ':' in title or title in seen:
                    continue
                idiom_url = "https://en.wiktionary.org" + a.get('href')
                seen.add(title)
                tasks.append(scrape_idiom_page(session, idiom_url, lang_code, counter))
                counter += 1

            results = await asyncio.gather(*tasks)
            results = [r for r in results if r]
            idioms.extend(results)

            next_link = soup.find("a", string="next page")
            page_url = f"https://en.wiktionary.org{next_link['href']}" if next_link else None

            await asyncio.sleep(1 + random.uniform(0, 0.5))

            if len(idioms) >= limit:
                break

    return idioms


def save_idioms_jsonl(idioms, language, output_dir='idioms_structured'):
    os.makedirs(output_dir, exist_ok=True)
    path = os.path.join(output_dir, f"seed_idioms_{language}.jsonl")
    with open(path, 'w', encoding='utf-8') as f:
        for item in idioms:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
    print(f"✅ Saved {len(idioms)} structured idioms to {path}")


In [10]:
from urllib.parse import unquote
import json
import re
def extract_inline_example(gloss):
    """Extract inline example from gloss if available."""
    if '“' in gloss or '"' in gloss:
        start = gloss.find('“') if '“' in gloss else gloss.find('"')
        end = gloss.find('”', start) if '”' in gloss else gloss.find('"', start + 1)
        if start != -1 and end != -1:
            return gloss[start + 1:end].strip()
    return ""

def clean_idioms(path):
    cleaned = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            data = json.loads(line)
            data['idiom'] = unquote(data['idiom'])

            # Extract inline example if possible
            if not data['example']:
                gloss = data.get('gloss', '')
                if '“' in gloss or '"' in gloss:
                    data['example'] = extract_inline_example(gloss)
            cleaned.append(data)

    with open(path.replace('.jsonl', '_cleaned.jsonl'), 'w', encoding='utf-8') as f:
        for item in cleaned:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

    print(f"✅ Cleaned and saved to {path.replace('.jsonl', '_cleaned.jsonl')}")

In [11]:
import nest_asyncio
import asyncio

nest_asyncio.apply()  # Permite múltiples loops en Jupyter

idioms = await scrape_wiktionary_idioms_structured("Telugu idioms", "te", limit=30000)
save_idioms_jsonl(idioms, "te")

✅ Saved 103 structured idioms to idioms_structured/seed_idioms_te.jsonl


In [12]:
clean_idioms("idioms_structured/seed_idioms_te.jsonl")

✅ Cleaned and saved to idioms_structured/seed_idioms_te_cleaned.jsonl


In [6]:
Idioms_Slang_Telugu = scrape_wiktionary_idioms_structured("Telugu slang", "te", limit=500)
save_idioms_jsonl(Idioms_Slang_Telugu, "te", output_dir='idioms_structured_temp')

TypeError: 'coroutine' object is not iterable

In [17]:
Idioms_Telugu_Telugulo = scrape_wiktionary_idioms_structured("తెలుగు జాతీయాలు", "te", limit=500)
save_idioms_jsonl(Idioms_Telugu_Telugulo, "te", output_dir='idioms_structured_temp')

✅ Saved 0 structured idioms to idioms_structured_temp/seed_idioms_te.jsonl


In [5]:
import nest_asyncio
import asyncio

nest_asyncio.apply()  # Permite múltiples loops en Jupyter

idioms = await scrape_wiktionary_idioms_structured("Spanish idioms", "es", limit=30000)
save_idioms_jsonl(idioms, "es")

✅ Saved 3179 structured idioms to idioms_structured/seed_idioms_es.jsonl


In [6]:
clean_spanish_idioms("idioms_structured/seed_idioms_es.jsonl")

✅ Cleaned and saved to idioms_structured/seed_idioms_es_cleaned.jsonl


In [11]:
Idioms_Hindi = scrape_wiktionary_idioms_structured("Hindi idioms", "hi", limit=500)

✅ Saved 99 structured idioms to idioms_structured_temp/seed_idioms_hi.jsonl
📅 Autosaved 100 idioms...


In [12]:
Idioms_Indonesian = scrape_wiktionary_idioms_structured("Indonesian idioms", "id", limit=500)
save_idioms_jsonl(Idioms_Indonesian, "id", output_dir='idioms_structured_temp')

✅ Saved 44 structured idioms to idioms_structured_temp/seed_idioms_id.jsonl


In [3]:
import nest_asyncio
import asyncio

nest_asyncio.apply()  # Permite múltiples loops en Jupyter

idioms = await scrape_wiktionary_idioms_structured("English idioms", "en", limit=30000)
save_idioms_jsonl(idioms, "en")


❌ Error processing idiom 'https://en.wiktionary.org/wiki/vapor': 
✅ Saved 9636 structured idioms to idioms_structured/seed_idioms_en.jsonl


In [13]:
clean_idioms("idioms_structured/seed_idioms_en.jsonl")

✅ Cleaned and saved to idioms_structured/seed_idioms_en_cleaned.jsonl
